# 006 · Abstract Classes and Protocols

> **Syntax › Classes & OOP** · Python 3.14

Python has no `interface` keyword. Contracts are expressed in two ways: **abstract base classes** (`abc.ABC`), which work like C# abstract classes, and **protocols** (`typing.Protocol`), which describe a shape that any class can satisfy *without declaring it* — formalised duck typing.

**In this notebook you will learn:**

- `abc.ABC` and `@abstractmethod`, and what happens when you instantiate an abstract class
- how to model C# interfaces with an ABC or with `typing.Protocol`
- structural typing: satisfying a protocol without "implements"
- `@runtime_checkable` for `isinstance` checks against protocols
- default method implementations (like C# default interface methods)
- when to choose an ABC, a protocol, or plain duck typing

## C# mental model

| C# | Python |
|---|---|
| `abstract class Shape` | `class Shape(ABC):` |
| `public abstract double Area();` | `@abstractmethod def area(self): ...` |
| `interface IShape` | an ABC with only abstract methods, **or** a `Protocol` |
| `class Circle : IShape` (nominal — must declare) | ABC: `class Circle(Shape)`; Protocol: nothing to declare |
| default interface method (C# 8+) | regular (non-abstract) method on the ABC/Protocol |
| `x is IShape` | `isinstance(x, Shape)` (Protocol needs `@runtime_checkable`) |

The key mental shift: C# typing is **nominal** — a class satisfies `IShape` only if it says so. Python is fundamentally **duck typed** ("if it walks like a duck and quacks like a duck…"): code calls `obj.area()` and it works if the method exists. ABCs add nominal checks on top; Protocols let *type checkers* verify duck typing **structurally**.

## Duck typing — the baseline

Before any formal contract, remember that Python does not need one for code to run. Any object with the right methods works:

In [ ]:
class Circle:
    def __init__(self, r):
        self.r = r

    def area(self):
        return 3.14159 * self.r ** 2

class Square:
    def __init__(self, s):
        self.s = s

    def area(self):
        return self.s ** 2

def total_area(shapes):
    return sum(s.area() for s in shapes)   # no interface needed

print(total_area([Circle(1), Square(2)]))

The downside: nothing documents or verifies the contract until a call fails at runtime. ABCs and Protocols fix that.

## Abstract base classes — `abc.ABC`

Inherit from `ABC` and decorate methods with `@abstractmethod`. Python then **refuses to instantiate** the class (or any subclass) until every abstract method is overridden — raising `TypeError` at runtime where C# gives a compile error.

```csharp
// C#
public abstract class Shape
{
    public abstract double Area();
    public string Describe() => $"{GetType().Name} with area {Area():F2}";
}
```

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        """Return the area of the shape."""

    def describe(self):                 # concrete method — inherited as-is
        return f"{type(self).__name__} with area {self.area():.2f}"

try:
    Shape()
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

An abstract method body can be just a docstring (or `...`). Subclasses become concrete once they implement every abstract method:

In [ ]:
class Rectangle(Shape):
    def __init__(self, w, h):
        self.w, self.h = w, h

    def area(self):
        return self.w * self.h

class Incomplete(Shape):
    pass                                # forgot area()

print(Rectangle(2, 3).describe())

try:
    Incomplete()
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** the check happens at **instantiation**, not when the class is defined. A subclass that forgets a method is only caught when someone tries to create it (type checkers catch it earlier).

Abstract properties, class methods and static methods are also supported — stack `@abstractmethod` as the **innermost** decorator:

In [ ]:
class Named(ABC):
    @property
    @abstractmethod
    def name(self) -> str: ...

class Product(Named):
    @property
    def name(self):
        return "lamp"

print(Product().name)

## ABCs as interfaces

A C# interface maps naturally onto an ABC containing only abstract methods. Since Python has multiple inheritance, a class can "implement" several of them:

```csharp
// C#
public interface IReadable { string Read(); }
public interface IWritable { void Write(string data); }
public class MemoryFile : IReadable, IWritable { ... }
```

In [ ]:
class Readable(ABC):
    @abstractmethod
    def read(self) -> str: ...

class Writable(ABC):
    @abstractmethod
    def write(self, data: str) -> None: ...

class MemoryFile(Readable, Writable):
    def __init__(self):
        self._data = ""

    def read(self):
        return self._data

    def write(self, data):
        self._data += data

f = MemoryFile()
f.write("hello")
print(f.read(), isinstance(f, Readable), isinstance(f, Writable))

## Protocols — structural typing

`typing.Protocol` defines an interface by **shape**. A class satisfies a protocol simply by having matching methods/attributes — it does **not** inherit from it or mention it. Static type checkers (mypy, pyright) verify the match. This is like Go interfaces or TypeScript's structural types; C# has no direct equivalent (the closest is how `foreach` or `await` accept any type with the right `GetEnumerator`/`GetAwaiter` method).

In [ ]:
from typing import Protocol

class SupportsArea(Protocol):
    def area(self) -> float: ...

class Triangle:                        # note: no base class, no "implements"
    def __init__(self, b, h):
        self.b, self.h = b, h

    def area(self) -> float:
        return 0.5 * self.b * self.h

def largest(shapes: list[SupportsArea]) -> float:
    return max(s.area() for s in shapes)

print(largest([Triangle(4, 3), Square(3), Circle(1)]))   # all satisfy SupportsArea

The type hint `list[SupportsArea]` now *documents and checks* the duck-typed contract. If you passed an object without an `area()` method, a type checker would report it — but at runtime, Protocols add **no** checks.

The standard library uses protocols widely: `collections.abc.Iterable`, `Sized`, `Hashable`, and `typing.SupportsInt`, `SupportsAbs`, etc.

## `@runtime_checkable` — `isinstance` with protocols

By default you cannot use a protocol with `isinstance`. Decorate it with `@runtime_checkable` to allow that check:

In [ ]:
from typing import runtime_checkable

@runtime_checkable
class Closeable(Protocol):
    def close(self) -> None: ...

class Connection:
    def close(self):
        print("closed")

print(isinstance(Connection(), Closeable))   # True — structural match
print(isinstance("text", Closeable))         # False

try:
    isinstance(Triangle(1, 1), SupportsArea)  # not runtime_checkable
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** a runtime check only verifies that the **names exist** — not their signatures or return types. An object with a `close` attribute of any kind passes:

In [ ]:
class Weird:
    close = 42          # not even a method

print(isinstance(Weird(), Closeable))   # True!

## Default method implementations

C# 8 added default interface methods. In Python, both ABCs and Protocols can simply contain ordinary methods with bodies. With an ABC (or when you **explicitly** subclass a Protocol), subclasses inherit them:

```csharp
// C#
public interface IGreeter
{
    string Name { get; }
    string Greet() => $"Hello, {Name}!";   // default implementation
}
```

In [ ]:
class Greeter(ABC):
    @property
    @abstractmethod
    def name(self) -> str: ...

    def greet(self):                    # default implementation
        return f"Hello, {self.name}!"

class EnglishGreeter(Greeter):
    @property
    def name(self):
        return "alice"

class LoudGreeter(EnglishGreeter):
    def greet(self):                    # override the default
        return super().greet().upper()

print(EnglishGreeter().greet())
print(LoudGreeter().greet())

⚠️ **Gotcha:** a class that satisfies a Protocol *structurally* (without subclassing it) does **not** get the protocol's default methods — it isn't actually a subclass. Inherit from the Protocol explicitly if you want them.

## Registering virtual subclasses

ABCs can also accept classes you don't control via `register()`, making `isinstance` return `True` without inheritance. No C# equivalent — it's mostly used by library authors:

In [ ]:
class Drawable(ABC):
    @abstractmethod
    def draw(self): ...

Drawable.register(str)             # declare that str "is a" Drawable
print(isinstance("hello", Drawable))

## When to use which?

| Use… | When |
|---|---|
| **Plain duck typing** | small scripts, internal code, obvious contracts |
| **`typing.Protocol`** | you want type-checked contracts *without* forcing callers to inherit — great for library inputs, third-party types, testing fakes. The default choice for "interfaces" in modern Python |
| **`abc.ABC`** | you own the hierarchy, want **runtime** enforcement (can't instantiate an incomplete class), and/or want to share implementation via inheritance — a C# abstract class |
| **`@runtime_checkable` Protocol** | you need `isinstance` dispatch on shape (accepting its limitations) |

## Summary

| Concept | C# | Python |
|---|---|---|
| Abstract class | `abstract class` | `class X(ABC):` |
| Abstract member | `abstract` method/property | `@abstractmethod` |
| Instantiating an abstract class | compile error | `TypeError` at runtime |
| Interface | `interface IX` | ABC with abstract methods, or `Protocol` |
| Implementing | `class C : IX` (required) | ABC: inherit; Protocol: just match the shape |
| Default interface method | method body in interface | ordinary method on the ABC/Protocol |
| Type test | `x is IX` | `isinstance(x, X)` (Protocol: `@runtime_checkable`) |
| Typing discipline | nominal | duck typing + nominal (ABC) or structural (Protocol) |

## Exercises

1. Write an ABC `Notifier` with an abstract method `send(message)` and a concrete method `send_all(messages)` that calls `send` for each. Implement `ConsoleNotifier`. Show that `Notifier()` raises `TypeError`.
2. Define a `Protocol` called `HasLength` requiring `__len__`. Write `def describe(x: HasLength) -> str` returning `f"length {len(x)}"`, and call it with a list, a string and a custom class.
3. Make `HasLength` runtime-checkable and test `isinstance` against `[1, 2]`, `42`, and `"abc"`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from abc import ABC, abstractmethod
from typing import Protocol, runtime_checkable

# 1
class Notifier(ABC):
    @abstractmethod
    def send(self, message): ...

    def send_all(self, messages):
        for m in messages:
            self.send(m)

class ConsoleNotifier(Notifier):
    def send(self, message):
        print(f"[notify] {message}")

ConsoleNotifier().send_all(["build ok", "deploy ok"])
try:
    Notifier()
except TypeError as e:
    print(e)

# 2, 3
@runtime_checkable
class HasLength(Protocol):
    def __len__(self) -> int: ...

class Playlist:
    def __init__(self, songs):
        self.songs = songs
    def __len__(self):
        return len(self.songs)

def describe(x: HasLength) -> str:
    return f"length {len(x)}"

print(describe([1, 2]), describe("abc"), describe(Playlist(["a"])))
print(isinstance([1, 2], HasLength), isinstance(42, HasLength), isinstance("abc", HasLength))
# True False True
```

</details>